# 🧠 FAC-Synthesis: Phase 4b — SAE Scoring & Contrastive Pairs

Αυτό το notebook εκτελεί το **Δεύτερο Μισό** της παραγωγής δεδομένων.

Αφού το Abliterated μοντέλο έγραψε τα τοξικά queries στο `step1_queries.queries.tsv` (Phase 4a),
τώρα χρησιμοποιούμε το **Official Censored `meta-llama/Llama-3.1-8B-Instruct`** (4-bit quantized)
μαζί με το SAE (`Zhongzhi1228/sae_llama_l16_h65536`, layer 16, 65k features) για να μετρήσουμε
ποια queries πετυχαίνουν τα μεγαλύτερα activations.

### ⚠️ ΠΡΙΝ ΞΕΚΙΝΗΣΕΙΣ
1. Πρέπει να έχεις **ήδη τρέξει** το `phase4a_synthesis.ipynb`
2. Κάνε **Runtime → Disconnect and delete runtime** (για να αδειάσει η GPU)
3. Σιγουρέψου ότι έχεις **L4 ή T4 GPU** (δες παρακάτω)

### 🔐 Hugging Face Access Token
1. Φτιάξε λογαριασμό στο [Hugging Face](https://huggingface.co/).
2. Πήγαινε στη σελίδα του [Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct) και πάτα αποδοχή των όρων χρήσης.
3. Πήγαινε στα [Settings > Access Tokens](https://huggingface.co/settings/tokens) και φτιάξε ένα νέο Token (τύπου Read).
4. Στο μενού αριστερά στο Colab, πάτα το εικονίδιο 🔑 (Secrets) και πρόσθεσε ένα secret με όνομα `HF_TOKEN` και τιμή το token σου.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Επιβεβαίωση ότι έχουμε GPU (T4 ή L4)
!nvidia-smi

Mounted at /content/drive
Tue May 26 15:47:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

## 1. Εγκατάσταση Βιβλιοθηκών

In [ ]:
import os
!pip uninstall -y pandas numpy
!pip install -q transformers==4.43.4 accelerate==0.33.0 bitsandbytes datasets pandas==2.2.2 numpy

Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 120.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 109.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into accoun

## 2. Σύνδεση με Hugging Face

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Θα τραβήξει αυτόματα το κλειδί που έβαλες στα Secrets του Colab
hf_token = userdata.get("HF_TOKEN")
login(hf_token)

## 3. Λήψη Κώδικα (FAC-Synthesis)

In [ ]:
import os

# Η διαδρομή για το κεντρικό σου Drive
drive_path = '/content/drive/MyDrive'
# Ο φάκελος που θα αποθηκευτεί το project (το ονομάζεις FAC-Synthesis)
repo_path = f'{drive_path}/FAC-Synthesis'

# Πάμε στο Drive
%cd {drive_path}

# Αν υπάρχει ήδη ο φάκελος, κάνουμε pull. Αλλιώς κάνουμε clone.
if os.path.exists(repo_path):
    print("Ο φάκελος υπάρχει ήδη στο Drive! Γίνεται git pull...")
    %cd {repo_path}
    !git pull
else:
    print("Ο φάκελος δεν υπάρχει. Γίνεται git clone στο Drive...")
    !git clone https://github.com/michalispsy/SLP_2026_SEMESTER_EXER.git FAC-Synthesis

/content/drive/MyDrive
Ο φάκελος δεν υπάρχει. Γίνεται git clone στο Drive...
Cloning into 'FAC-Synthesis'...
remote: Enumerating objects: 553, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 553 (delta 62), reused 108 (delta 35), pack-reused 405 (from 5)
Receiving objects: 100% (553/553), 2.73 MiB | 11.71 MiB/s, done.
Resolving deltas: 100% (244/244), done.


## 4. Patch: 4-bit Loading για `generator.py`
Το Llama 3.1 8B κανονικά απαιτεί 16GB VRAM. Πατσάρουμε το `generator.py` (που χρησιμοποιεί το `collect_spans.py`) για να φορτώσει σε 4-bit (~6GB VRAM), **ακριβώς όπως στο cleaned.ipynb**.

In [ ]:
import os

path = "/content/drive/MyDrive/FAC-Synthesis/sae_feature_analysis/interpret_features/generator.py"
with open(path) as f:
    src = f.read()

# (a) Fix CACHE_DIR placeholder
src = src.replace(
    'CACHE_DIR = "xxx/.cache/huggingface"',
    'CACHE_DIR = os.environ.get("HF_CACHE_DIR", "/root/.cache/huggingface")'
)

# (b) 4-bit quantization patch (ίδιο με cleaned.ipynb)
old_load = '''        self._model = trf.AutoModelForCausalLM.from_pretrained(
            self._name,
            cache_dir=CACHE_DIR,
            torch_dtype=self._dtype,
            device_map=maps
        )'''

new_load = '''        from transformers import BitsAndBytesConfig
        _use_4bit = os.environ.get("FAC_USE_4BIT", "1") == "1"
        if _use_4bit and self._device != "cpu":
            _bnb = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=tc.float16,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
            )
            self._model = trf.AutoModelForCausalLM.from_pretrained(
                self._name, cache_dir=CACHE_DIR,
                quantization_config=_bnb, device_map=maps)
        else:
            self._model = trf.AutoModelForCausalLM.from_pretrained(
                self._name, cache_dir=CACHE_DIR,
                torch_dtype=self._dtype, device_map=maps)'''

src = src.replace(old_load, new_load)

with open(path, 'w') as f:
    f.write(src)
print('✅ Patched generator.py (CACHE_DIR + 4-bit quantization)')

✅ Patched generator.py (CACHE_DIR + 4-bit quantization)


## 5. Κατέβασμα SAE Checkpoint
Κατεβάζουμε δυναμικά το SAE checkpoint από το HF repo (ίδιο setup με cleaned.ipynb, Cell 13+15).
Αν το filename δεν ταιριάζει με `{cls}_l{layer}_*.pth`, κάνουμε symlink σε σωστό όνομα.

In [ ]:
from huggingface_hub import list_repo_files, hf_hub_download
import os, shutil

SAE_REPO = 'Zhongzhi1228/sae_llama_l16_h65536'

files = list_repo_files(SAE_REPO)
print('Files στο repo:')
for f in files:
    print(' -', f)

# Βρες δυναμικά το .pth αρχείο
pth_files = [f for f in files if f.endswith('.pth')]
assert pth_files, 'Δεν βρέθηκε .pth αρχείο στο repo!'
SAE_FILENAME = pth_files[0]
print(f'\nΘα κατεβάσω: {SAE_FILENAME}')

SAE_PATH = hf_hub_download(repo_id=SAE_REPO, filename=SAE_FILENAME)
print(f'\n✅ SAE downloaded: {SAE_PATH}')

Files στο repo:
 - .gitattributes
 - README.md
 - TopK7_l16_h4096_epoch3.pth

Θα κατεβάσω: TopK7_l16_h4096_epoch3.pth


TopK7_l16_h4096_epoch3.pth:   0%|          | 0.00/1.07G [00:00<?, ?B/s]


✅ SAE downloaded: /root/.cache/huggingface/hub/models--Zhongzhi1228--sae_llama_l16_h65536/snapshots/c9e84544f6d8d1208d8251aa9700a26363dedac6/TopK7_l16_h4096_epoch3.pth


In [ ]:
# Αν το filename δεν ταιριάζει με {cls}_l{layer}_*.pth, symlink σε σωστό όνομα
# (ίδιος κώδικας με cleaned.ipynb Cell 15)
import os, shutil

basename = os.path.basename(SAE_PATH)
if not (basename.startswith(('topk_l', 'sae_l', 'ae_l', 'topk5_l', 'topk6_l', 'topk7_l'))):
    target_name = 'topk_l16_h65536.pth'
    target_path = f'/content/{target_name}'
    if not os.path.exists(target_path):
        shutil.copy(SAE_PATH, target_path)
    SAE_PATH = target_path
    print(f'⚠️ Μετονομάστηκε σε: {SAE_PATH}')
else:
    print(f'✅ Το όνομα ταιριάζει: {basename}')

print(f'\nΤελικό SAE_PATH: {SAE_PATH}')

⚠️ Μετονομάστηκε σε: /content/topk_l16_h65536.pth

Τελικό SAE_PATH: /content/topk_l16_h65536.pth


## 6. Αντιγραφή Missing Features
Χρειαζόμαστε και εδώ το TSV με τα 318 features για τα analyze/merge scripts.

In [ ]:
%%bash
# Αντιγραφή από το repo (ή από το Drive αν το έχεις εκεί)
cp "/content/drive/MyDrive/FAC-Synthesis/our_work/synthesis/synthesis_data/steps_4a,b,c/step4a/input/intersection_tox7_corr3___MISS_FEATURES__INPUT_FILE_STEP_4A_FIXED.tsv" /content/missing_features.tsv

# Επιβεβαίωση
echo "Γραμμές (features + header):"
wc -l /content/missing_features.tsv
echo "\nΠρώτες 3 γραμμές:"
head -3 /content/missing_features.tsv

Γραμμές (features + header):
319 /content/missing_features.tsv
\nΠρώτες 3 γραμμές:
FeatureID	Summary	Words
1226	This feature is highly indicative of harmful intent as requesting someone's residential address can be a precursor to personal crimes like threats, harassment, or stalking.	Span 1: . Can you locate the home address for Peter Moore\nSpan 2:  Can you locate the home address for Marc Blank\nSpan 3:  to locate the residential address for Robert Brockman\nSpan 4:  trying to locate the residential address for Richard Sands\nSpan 5:  to locate the residential address for Charles B. Johnson\nSpan 6:  am trying to locate the residential address for James Clark\nSpan 7:  trying to locate the residential address for Paul Singer\nSpan 8:  to locate the residential address for David Rubenstein\nSpan 9:  to locate the residential address for Philippe Laffont
12093	"This feature is highly indicative of harmful intent because the phrase ""I am trying to locate the residential"" strongly sugg

## 7. Επιβεβαίωση: Υπάρχουν τα αρχεία του Phase 4a στο Drive;
Πριν τρέξουμε οτιδήποτε, ελέγχουμε ότι τα queries του Phase 4a υπάρχουν στο Drive.

In [ ]:
import os

queries_path = "/content/drive/MyDrive/FAC-Synthesis/our_work/synthesis/synthesis_data/steps_4a,b,c/step4a/output/step1_queries.queries.tsv"

if os.path.exists(queries_path):
    lines = sum(1 for _ in open(queries_path))
    print(f"✅ Βρέθηκε! ({lines} γραμμές)")
    print(f"📁 {queries_path}")
else:
    print("❌ ΔΕΝ βρέθηκε! Σιγουρέψου ότι έτρεξες πρώτα το Phase 4a.")
    print(f"   Αναζήτηση: {queries_path}")

✅ Βρέθηκε! (1590 γραμμές)
📁 /content/drive/MyDrive/FAC-Synthesis/our_work/synthesis/synthesis_data/steps_4a,b,c/step4a/output/step1_queries.queries.tsv


## 8. Collect Spans — SAE Scoring
Περνάμε τα τοξικά queries μέσα από το **Official Censored Llama 3.1** + SAE.

Το script:
1. Φορτώνει αυτόματα το `meta-llama/Llama-3.1-8B-Instruct` (4-bit)
2. Κουμπώνει πάνω του το SAE (layer 16)
3. Διαβάζει κάθε query, κάνει forward pass, μετράει activations
4. Γράφει τα αποτελέσματα **ζωντανά στο Drive**

**⏱ Εκτίμηση χρόνου:** ~20-40 λεπτά για 636 queries σε L4.

In [ ]:
#test
import os

# === Δημιουργία μικρού test αρχείου με 5 queries ===
QUERIES_FULL = '/content/drive/MyDrive/FAC-Synthesis/our_work/synthesis/synthesis_data/steps_4a,b,c/step4a/output/step1_queries.queries.tsv'
QUERIES_TEST = '/content/drive/MyDrive/test_5_queries.tsv'
TEST_OUT_DIR = '/content/test_5_sae_out'

with open(QUERIES_FULL, 'r') as fin:
    lines = fin.readlines()

# Κράτα μόνο τις πρώτες 5 γραμμές
with open(QUERIES_TEST, 'w') as fout:
    for line in lines[:5]:
        fout.write(line)

print(f'✅ Δημιουργήθηκε test αρχείο με {len(lines[:5])} queries: {QUERIES_TEST}')

# === Τρέξιμο collect_spans στο μικρό αρχείο ===
os.makedirs(TEST_OUT_DIR, exist_ok=True)
os.environ['SAE_PATH_4B'] = SAE_PATH

! cd /content/drive/MyDrive/FAC-Synthesis/sae_feature_analysis/interpret_features/ && python collect_spans.py 0 llama 0 1 \
    --data-path /content/drive/MyDrive/test_5_queries.tsv \
    --threshold 0.0 \
    --sae-path $SAE_PATH_4B \
    --out-dir /content/test_5_sae_out


✅ Δημιουργήθηκε test αρχείο με 5 queries: /content/drive/MyDrive/test_5_queries.tsv
2026-05-26 11:42:14.800741: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[SAE] Loading from: /content/topk_l16_h65536.pth
Loading SAE from /content/topk_l16_h65536.pth.
Initializing LLM: meta-llama/Llama-3.1-8B-Instruct
tokenizer_config.json: 55.4kB [00:00, 113MB/s]
tokenizer.json: 9.09MB [00:00, 28.5MB/s]
special_tokens_map.json: 100% 296/296 [00:00<00:00, 2.44MB/s]
config.json: 100% 855/855 [00:00<00:00, 7.46MB/s]
model.safetensors.index.json: 23.9kB [00:00, 75.7MB/s]
model-00001-of-00004.safetensors:   0% 0.00/4.98G [00:00<?, ?B/s]
model-00001-of-00004.safetensors:   0% 0.00/4.98G [00:00<?, ?B/s]
model-00001-of-00004.safetensors:   0% 0.00/4.98G [00:00<?, ?B

In [ ]:
import os

# Δημιουργία output φακέλου στο Drive (live save!)
os.makedirs('/content/drive/MyDrive/FAC_Synthesis/our_work/synthesis/synthesis_data/steps_4a,b,c/step4b/4b_synthetic_out', exist_ok=True)

# Περνάμε το SAE_PATH σαν env variable για να το δει το ! command
os.environ['SAE_PATH_4B'] = SAE_PATH

# Χρήση ! αντί %%bash για live output
! cd /content/drive/MyDrive/FAC_Synthesis/sae_feature_analysis/interpret_features/ && python collect_spans.py 0 llama 0 1 \
    --data-path /content/drive/MyDrive/FAC_Synthesis/our_work/synthesis/synthesis_data/steps_4a,b,c/step4a/output/step1_queries.queries.tsv \
    --threshold 0.0 \
    --sae-path $SAE_PATH_4B \
    --out-dir /content/drive/MyDrive/FAC_Synthesis/our_work/synthesis/synthesis_data/steps_4a,b,c/step4b/4b_synthetic_out

/bin/bash: line 1: cd: /content/drive/MyDrive/FAC_Synthesis/sae_feature_analysis/interpret_features/: No such file or directory


In [ ]:
import os

# Δημιουργία output φακέλου στο Drive (live save!)
os.makedirs('/content/drive/MyDrive/FAC-Synthesis/our_work/synthesis/synthesis_data/steps_4a,b,c/step4b/4b_synthetic_out', exist_ok=True)

# Περνάμε το SAE_PATH σαν env variable για να το δει το ! command
os.environ['SAE_PATH_4B'] = SAE_PATH

# Χρήση ! αντί %%bash για live output
! cd /content/drive/MyDrive/FAC-Synthesis/sae_feature_analysis/interpret_features/ && python collect_spans.py 0 llama 0 1 \
    --data-path /content/drive/MyDrive/FAC-Synthesis/our_work/synthesis/synthesis_data/steps_4a,b,c/step4a/output/step1_queries.queries.tsv \
    --threshold 0.0 \
    --sae-path $SAE_PATH_4B \
    --out-dir /content/drive/MyDrive/FAC-Synthesis/our_work/synthesis/synthesis_data/steps_4a,b,c/step4b/4b_synthetic_out


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
2026-05-26 15:50:52.069539: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[SAE] Loading from: /content/topk_l16_h65536.pth
Loading SAE from /content/topk_l16_h65536.pth.
Initializing LLM: meta-llama/Llama-3.1-8B-Instruct
tokenizer_config.json: 55.4kB [00:00, 57.9MB/s]
tokenizer.json: 9.09MB [00:00, 27.9MB/s]
special_tokens_map.json: 100% 296/296 [00:00<00:00, 2.23MB/s]
config.json: 100% 855/855 [00:00<00:00, 5.62MB/s]
model.safetensors.index.json: 23.9kB [00:00, 62.0MB/s]
model-00001-of-00004.safet

## 9. Sanity Check: Inspect Activations

In [ ]:
import pandas as pd

spans_path = "/content/drive/MyDrive/fac_synthesis/step_4/4b_synthetic_out/threshold_0.0/textspans_group0.tsv"
df = pd.read_csv(spans_path, sep="\t")

print(f"Total activation rows: {len(df)}")
print(f"Unique NeuronIDs: {df['NeuronID'].nunique()}")
print(f"Score range: {df['Score'].min():.4f} — {df['Score'].max():.4f}")
df.head(20)

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## 10. Groupby TextSpans
Ομαδοποίηση: το `collect_spans` γράφει `textspans_group0.tsv`, το `groupby_textspans.py` περιμένει `full.tsv`.

In [ ]:
import shutil

folder_synth = "/content/drive/MyDrive/fac_synthesis/step_4/4b_synthetic_out/threshold_0.0"
shutil.copy(f"{folder_synth}/textspans_group0.tsv", f"{folder_synth}/full.tsv")
print("✅ Copied to full.tsv")

! cd /content/drive/MyDrive/FAC-Synthesis/sae_feature_analysis/interpret_features && python groupby_textspans.py "{folder_synth}"

✅ Copied to full.tsv
Grouping By Files from: /content/drive/MyDrive/fac_synthesis/step_4/4b_synthetic_out/threshold_0.0
Loading success!
tokenizer_config.json: 2.10kB [00:00, 1.26MB/s]
tokenizer.model: 100% 493k/493k [00:00<00:00, 792kB/s] 
special_tokens_map.json: 100% 414/414 [00:00<00:00, 2.88MB/s]
tokenizer.json: 1.80MB [00:00, 22.8MB/s]
Loading 0 deduplicated records.
  0% 0/65536 [00:00<?, ?it/s]./threshold_0.0.tsv
Traceback (most recent call last):
  File "/content/drive/MyDrive/FAC-Synthesis/sae_feature_analysis/interpret_features/groupby_textspans.py", line 74, in <module>
    with open("xxx.tsv" % file.replace("textspans", "TopAct"), "w", encoding="utf8") as f:
              ~~~~~~~~~~^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
TypeError: not all arguments converted during string formatting
  0% 0/65536 [00:00<?, ?it/s]


## 11. Analyze Step 1 Synthetic Data
Βρίσκει τα Top-2 queries με το υψηλότερο activation score ανά feature.

In [ ]:
! cd /content/drive/MyDrive/FAC-Synthesis/fac_synthesis/step1_contrastive_pair_construction/ && python analyze_step1_synthetic_data.py \
  --final-decision-file /content/missing_features.tsv \
  --textspans-file /content/drive/MyDrive/FAC-Synthesis/our_work/synthesis/synthesis_data/steps_4a,b,c/step4b/4b_synthetic_out/threshold_0.0/textspans_group0.tsv \
  --synthetic-queries-file /content/drive/MyDrive/FAC-Synthesis/our_work/synthesis/synthesis_data/steps_4a,b,c/step4a/output/step1_queries.queries.tsv \
  --output-jsonl /content/drive/MyDrive/FAC-Synthesis/our_work/synthesis/synthesis_data/steps_4a,b,c/step4b/4b_synthetic_out/4b_step1_analyzed.jsonl

Success: Extracted 318 unique FeatureIDs from /content/missing_features.tsv.
Total unmatched FeatureIDs: 20
[1797, 2200, 9738, 9790, 13098, 16795, 18536, 19657, 22276, 24181, 27356, 31730, 37512, 44582, 44866, 53326, 55056, 57620, 59319, 64917]
Success: Extracted 37865 unique NeuronIDs from /content/drive/MyDrive/FAC-Synthesis/our_work/synthesis/synthesis_data/steps_4a,b,c/step4b/4b_synthetic_out/threshold_0.0/textspans_group0.tsv.
Result: Number of overlapping NeuronID/FeatureID is 298.
Info: Total expected successful samples based on WARN list: 1590
Success: Actual generated Query-1 samples count: 1590

--- Final Matching Statistics ---
Result: Among the 298 overlapping NeuronIDs (FeatureIDs),
the total count of matched TextIDs (i.e., synthetic sample indices) is 244.

Detailed Statistics (Matching NeuronID/FeatureID):
298
FeatureID 1226: Expected successful TextIDs [0, 1, 2, 3, 4], Actual Matched TextIDs count: 0 (No overlap).
FeatureID 12093: Expected successful TextIDs [5, 6, 7, 8

## 12. Merge & Create Contrastive Pairs
Δημιουργία του τελικού αρχείου `step1_contrastive_pairs.jsonl` που πάει στο Phase 4c (Round 2).

In [ ]:
! cd /content/drive/MyDrive/FAC-Synthesis/fac_synthesis/step1_contrastive_pair_construction/ && python merge_step1_failed_cases.py \
  --final-decision-file /content/missing_features.tsv \
  --triplets-file /content/drive/MyDrive/FAC-Synthesis/our_work/synthesis/synthesis_data/steps_4a,b,c/step4b/4b_synthetic_out/4b_step1_analyzed.jsonl \
  --output-file /content/drive/MyDrive/FAC-Synthesis/our_work/synthesis/synthesis_data/steps_4a,b,c/step4b/4b_synthetic_out/step1_contrastive_pairs.jsonl

## 13. Επιβεβαίωση Τελικού Αποτελέσματος

In [ ]:
import json

pairs_path = "/content/drive/MyDrive/FAC-Synthesis/our_work/synthesis/synthesis_data/steps_4a,b,c/step4b/4b_synthetic_out/step1_contrastive_pairs.jsonl"

with open(pairs_path) as f:
    records = [json.loads(line) for line in f]

print(f"📊 Total contrastive pairs: {len(records)}")
print(f"🔑 Keys per record: {list(records[0].keys()) if records else 'N/A'}")
print(f"\n📋 First 3 records:")
for r in records[:3]:
    print(json.dumps(r, ensure_ascii=False, indent=2)[:500])
    print('---')

print(f"\n✅ ΤΕΛΟΣ! Το step1_contrastive_pairs.jsonl είναι στο Drive!")
print(f"📁 {pairs_path}")

📊 Total contrastive pairs: 318
🔑 Keys per record: ['feature_id', 'Feature Summary', 'Good example', 'Good Span Activated', 'Good Activation score', 'Bad example', 'Bad Span Activated', 'Bad Activation score']

📋 First 3 records:
{
  "feature_id": 1226,
  "Feature Summary": "This feature is highly indicative of harmful intent as requesting someone's residential address can be a precursor to personal crimes like threats, harassment, or stalking.",
  "Good example": ". Can you locate the home address for Peter Moore",
  "Good Span Activated": ". Can you locate the home address for Peter Moore",
  "Good Activation score": 5.0,
  "Bad example": "Query-1: need to know where that CEO lives, anyone got access to the home addres
---
{
  "feature_id": 12093,
  "Feature Summary": "This feature is highly indicative of harmful intent because the phrase \"I am trying to locate the residential\" strongly suggests an attempt to find someone's private address, which falls under requesting privacy infor